# 📊 Trực quan hóa kết quả Pipeline phân loại tài liệu
Notebook này đọc file JSON kết quả của pipeline và tạo các biểu đồ trực quan.

**File nguồn:** `output/pipeline/20260109-04.pipeline.json`

In [ ]:
import json
import sys
import os
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Đảm bảo src có thể import
ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cấu hình đường dẫn
JSON_PATH = ROOT / 'output' / 'pipeline' / '20260109-04.pipeline.json'

print(f'Đang đọc: {JSON_PATH}')
with open(JSON_PATH, encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ Đọc thành công! File: {data['file_name']}, Tổng trang: {data['total_pages']}")

## 1️⃣ Tổng quan kết quả pipeline

In [ ]:
fr = data['file_routing']

summary = pd.DataFrame([
    {'Chỉ số': 'Tên file', 'Giá trị': data['file_name']},
    {'Chỉ số': 'Tổng số trang', 'Giá trị': data['total_pages']},
    {'Chỉ số': 'Quyết định tổng thể', 'Giá trị': fr['decision'].upper()},
    {'Chỉ số': 'Tổng số lát cắt', 'Giá trị': fr['n_slices']},
    {'Chỉ số': '✅ Tự động (auto_file)', 'Giá trị': fr['n_auto']},
    {'Chỉ số': '⚠️ Cần người duyệt', 'Giá trị': fr['n_needs_human']},
    {'Chỉ số': 'Tỉ lệ tự động', 'Giá trị': f"{fr['n_auto']/fr['n_slices']*100:.1f}%"},
    {'Chỉ số': 'Bộ tách sử dụng', 'Giá trị': fr.get('splitter', 'N/A')},
])

display(summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

## 2️⃣ Biểu đồ: Auto vs Needs Human

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Pie chart ---
ax = axes[0]
labels = ['✅ Auto file', '⚠️ Needs human']
sizes  = [fr['n_auto'], fr['n_needs_human']]
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0)
ax.pie(sizes, labels=labels, colors=colors, explode=explode,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
ax.set_title('Tỉ lệ tự động vs Cần người', fontsize=14, fontweight='bold')

# --- Bar chart số lát cắt ---
ax2 = axes[1]
bars = ax2.bar(['Auto file', 'Needs human'], sizes, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sizes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', va='bottom', fontsize=14, fontweight='bold')
ax2.set_ylabel('Số lát cắt', fontsize=12)
ax2.set_title('Số lát cắt theo quyết định', fontsize=14, fontweight='bold')
ax2.set_ylim(0, fr['n_slices'] + 5)
ax2.spines[['top', 'right']].set_visible(False)

plt.suptitle(f"Kết quả Pipeline: {data['file_name']}", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'output' / 'pipeline_routing.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: output/pipeline_routing.png')

## 3️⃣ Phân loại theo loại tài liệu

In [ ]:
clfs = data.get('classifications', [])
routing = data.get('routing', [])

df = pd.DataFrame([
    {
        'slice_id': c['slice_id'],
        'loai_tai_lieu': c.get('predicted_category', 'UNKNOWN'),
        'tier': c.get('tier', '?'),
        'confidence': round(c.get('confidence') or 0, 4),
    }
    for c in clfs
])

# Gắn thêm quyết định routing
routing_map = {r['slice_id']: r['decision'] for r in routing}
df['decision'] = df['slice_id'].map(routing_map)

# Đếm số lát cắt mỗi loại
cat_counts = df['loai_tai_lieu'].value_counts().reset_index()
cat_counts.columns = ['Loại tài liệu', 'Số lát cắt']

fig, ax = plt.subplots(figsize=(12, 6))
colors_bar = ['#e74c3c' if c == 'UNKNOWN' else '#3498db' for c in cat_counts['Loại tài liệu']]
bars = ax.barh(cat_counts['Loại tài liệu'], cat_counts['Số lát cắt'],
               color=colors_bar, edgecolor='white', linewidth=1)
for bar, val in zip(bars, cat_counts['Số lát cắt']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Số lát cắt', fontsize=12)
ax.set_title('Phân bố loại tài liệu được phân loại', fontsize=14, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
blue_patch = mpatches.Patch(color='#3498db', label='Đã phân loại (auto)')
red_patch  = mpatches.Patch(color='#e74c3c', label='UNKNOWN (needs_human)')
ax.legend(handles=[blue_patch, red_patch], fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / 'output' / 'pipeline_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: output/pipeline_categories.png')

## 4️⃣ Biểu đồ: Phân loại theo phương pháp (Tier)

In [ ]:
tier_counts = df['tier'].value_counts()

tier_colors = {'rule': '#27ae60', 'tfidf': '#2980b9', 'llm': '#8e44ad', 'human': '#e74c3c'}
colors_tier = [tier_colors.get(t, '#95a5a6') for t in tier_counts.index]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tier_counts.index, tier_counts.values, color=colors_tier,
              edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, tier_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel('Số lát cắt', fontsize=12)
ax.set_title('Phương pháp phân loại (Tier)', fontsize=14, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

tier_labels = {
    'rule': 'Rule: Luật từ khóa',
    'tfidf': 'TF-IDF: Mô hình thống kê',
    'llm': 'LLM: Mô hình ngôn ngữ',
    'human': 'Human: Cần người duyệt'
}
patches = [mpatches.Patch(color=tier_colors.get(t, '#95a5a6'), label=tier_labels.get(t, t))
           for t in tier_counts.index]
ax.legend(handles=patches, fontsize=10, loc='upper right')
plt.tight_layout()
plt.savefig(ROOT / 'output' / 'pipeline_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: output/pipeline_tiers.png')

## 5️⃣ Biểu đồ: Độ tin cậy (Confidence) của từng lát cắt

In [ ]:
df_conf = df[df['tier'] != 'human'].copy()

fig, ax = plt.subplots(figsize=(14, 5))
colors_conf = ['#27ae60' if c >= 0.9 else '#f39c12' if c >= 0.5 else '#e74c3c'
               for c in df_conf['confidence']]
ax.bar(range(len(df_conf)), df_conf['confidence'], color=colors_conf, edgecolor='white')
ax.axhline(0.9, color='green', linestyle='--', linewidth=1.5, label='Ngưỡng cao (0.9)')
ax.axhline(0.5, color='orange', linestyle='--', linewidth=1.5, label='Ngưỡng thấp (0.5)')
ax.set_xticks(range(len(df_conf)))
ax.set_xticklabels(df_conf['slice_id'].str.replace('slice_', 'S'), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Confidence', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('Độ tin cậy từng lát cắt (chỉ lát đã phân loại được)', fontsize=14, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
green_p = mpatches.Patch(color='#27ae60', label='Rất tự tin (≥0.9)')
orange_p = mpatches.Patch(color='#f39c12', label='Trung bình (0.5–0.9)')
red_p = mpatches.Patch(color='#e74c3c', label='Thấp (<0.5)')
ax.legend(handles=[green_p, orange_p, red_p], fontsize=10)
plt.tight_layout()
plt.savefig(ROOT / 'output' / 'pipeline_confidence.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: output/pipeline_confidence.png')

## 6️⃣ Bảng chi tiết toàn bộ lát cắt

In [ ]:
def color_decision(val):
    if val == 'auto_file':   return 'background-color: #d4efdf; color: #1e8449'
    if val == 'needs_human': return 'background-color: #fadbd8; color: #c0392b'
    return ''

def color_tier(val):
    if val == 'rule':   return 'color: #27ae60; font-weight: bold'
    if val == 'tfidf':  return 'color: #2980b9; font-weight: bold'
    if val == 'llm':    return 'color: #8e44ad; font-weight: bold'
    if val == 'human':  return 'color: #e74c3c; font-weight: bold'
    return ''

def color_conf(val):
    if val >= 0.9:  return 'color: #27ae60; font-weight: bold'
    if val >= 0.5:  return 'color: #f39c12'
    if val > 0:     return 'color: #e74c3c'
    return 'color: gray'

display(
    df.rename(columns={
        'slice_id': 'Lát cắt',
        'loai_tai_lieu': 'Loại tài liệu',
        'tier': 'Phương pháp',
        'confidence': 'Độ tin cậy',
        'decision': 'Quyết định'
    }).style
    .applymap(color_decision, subset=['Quyết định'])
    .applymap(color_tier,     subset=['Phương pháp'])
    .applymap(color_conf,     subset=['Độ tin cậy'])
    .format({'Độ tin cậy': '{:.4f}'})
    .hide(axis='index')
)

print(f"\nTổng: {len(df)} lát cắt | Auto: {(df['decision']=='auto_file').sum()} | Needs human: {(df['decision']=='needs_human').sum()}")

## 7️⃣ Cảnh báo & Thời gian xử lý

In [ ]:
print('⚠️ CẢNH BÁO:')
for w in data.get('warnings', []):
    print(' •', w)

print('\n⏱️ THỜI GIAN XỬ LÝ:')
timings = data.get('stage_timings_ms', {})
for k, v in timings.items():
    print(f'  {k:20s}: {v:.1f} ms')

total = sum(v for v in timings.values() if isinstance(v, (int, float)))
print(f'  {"TỔNG":20s}: {total:.1f} ms')